==============================================================================
ESG RAG 인덱스 구축 스크립트 (Google Colab용) — v6 (프로덕션, 100개사 본작업용)

v5.2 -> v6 수정 사항:
  검증 단계에서 쓰던 셀들을 전부 제거하고, SK하이닉스로 하드코딩됐던 부분을
  일반화한 배포용 버전. 로직 자체는 v5.2(pdfplumber + PyMuPDF 2단계 캐스케이딩)와
  동일함 — 아래 변경은 전부 '무엇을 뺐는지/일반화했는지'에 대한 것이고 새 기능은 없음.

  삭제됨 (검증 전용, 100개사 본작업엔 불필요):
    - Cell Q2 (section_title 탐지 수동 검증)
    - Cell Q (표/텍스트 분리 수동 확인 도구) + 그걸 호출하던 테스트 셀
    - 'Cell 6 검증' (SK하이닉스 첫 문서로 시험 청킹하던 셀)
    - Cell 6b/6c (10개사 필터 + 진단 전용 실행 — PyMuPDF 캐스케이딩 검증용)

  일반화됨 (SK하이닉스 하드코딩 제거):
    - Cell 7b: 재인덱싱 대상 회사 목록을 하드코딩값 대신 직접 채워 넣는 변수로 변경
    - Cell 9: 동작 확인 테스트가 manifest의 첫 번째 회사를 자동으로 사용하도록 변경

  검증 이력이 필요하면 build_rag_index_v5_2_pymupdf_only.ipynb를 참고할 것
  (10개사 실측: DB하이텍류 1개사만 캐스케이딩 후에도 심각한 잔여 실패,
  나머지는 PyMuPDF로 대부분 복구됨 — 자세한 수치는 v5.2 히스토리 참고).

v5.1 -> v5.2 수정 사항:
  LlamaParse(3단계)가 페이지당 수초~수십초로 너무 느려서, 검증 단계에서는
  일단 제외. pdfplumber -> PyMuPDF 2단계 캐스케이딩만 남긴 버전.
  - Cell 5의 LlamaParse 관련 함수(llamaparse_rescue_page 등) 전체 삭제
  - Cell 6(process_document)의 3단계 호출부(주석 처리돼있던 부분) 삭제
  - Cell 1/2/3에서 LlamaParse API 키·pip 설치 관련 줄 삭제
  - [신규] Cell 4b 아래에 10개사 검증용 필터 + 진단 전용 실행 셀을 정식으로
    편입 (이전엔 채팅으로만 안내했던 걸 노트북에 반영). 임베딩/Chroma 적재
    없이 process_document만 호출해서 파싱 실패율만 무료로 확인 가능.

  이 파일은 검증 전용입니다. 10개사 결과 확인 후 문제 없으면, 검증용 셀들을
  다 빼고 LlamaParse 3단계(필요시)를 다시 붙인 최종 프로덕션 파일을 별도로
  만들 예정 (100개사 본작업용).

v5 -> v5.1 수정 사항 (10개사 실측 결과 반영):
  실측: pdfplumber 실패 페이지를 PyMuPDF로 재시도해보니, DB하이텍(000990)은
  92p 중 79p가 여전히 실패(진짜로 텍스트 레이어가 없는 벡터 윤곽선 폰트)했지만,
  경동나비엔(97% 복구)·가온전선(91% 복구)은 pdfplumber만 못 읽었을 뿐 PyMuPDF로
  거의 다 복구됨 -> 'vector_curve_text' 진단 라벨은 "텍스트가 없다"가 아니라
  "pdfplumber가 이 페이지를 못 읽는다"였을 뿐인 경우가 대부분.

  1. [Cell 5] PyMuPDF 캐스케이딩 추가 (무료, 항상 활성).
     extract_page_content가 4번째 값으로 table_bboxes를 반환하도록 변경.
     process_document가 extraction_issue 발생 시 자동으로 PyMuPDF 재시도.
     주의: PyMuPDF의 get_text()는 표 영역을 제외 안 해서 그대로 쓰면 표가
     마크다운 청크 + 뒤죽박죽 텍스트 청크로 이중 적재되는 v1급 버그가 재발함
     -> pymupdf_rescue_text()가 pdfplumber의 table_bboxes를 재사용해 표 영역과
        겹치는 블록을 제외하고 텍스트만 가져오도록 처리.
  2. [Cell 6] LlamaParse 3단계 폴백 코드 추가 — 단, 기본 비활성 (주석 처리).
     PyMuPDF로도 복구 안 되는 페이지(DB하이텍류, 실측상 전체의 ~10% 회사 추정)만
     대상으로 페이지 단위 호출하도록 설계 (문서 전체를 보내지 않아 크레딧 절감).
     활성화하려면 process_document 안의 주석 처리된 3단계 블록만 해제하면 됨
     (API 키는 Cell 2에서 이미 프롬프트로 물어보게 해둠 — 안 쓰면 그냥 Enter).
     비용 재추정: 100개사 확장 시 잔여 실패 약 2,800페이지 -> Cost-effective
     기준 약 8,400크레딧(≈$10.5) -> LlamaParse 무료 월 10,000크레딧 안에 들어옴.
  3. Tesseract OCR은 채택 안 함 (PyMuPDF 캐스케이딩만으로 스캔본 상당수도
     같이 복구되는 것을 확인했고, 남은 진짜 실패분은 LlamaParse 비용이
     무료 티어 안에 들어올 만큼 작아서 별도 OCR 엔진을 붙일 실익이 낮음).

v4 -> v5 수정 사항 (팀원 파이프라인 비교 리뷰 반영, 1·2번만 우선 반영):
  1. [Cell 5] 텍스트 추출 실패를 조용히 넘어가던 문제 확인 (jin의 DB하이텍 케이스:
     본문이 벡터 윤곽선(curve) 폰트라 92p 중 87p, 95% 텍스트 손실 — v4는 이걸
     감지도 못 하고 빈 텍스트로 통과시킴).
     -> extract_page_content가 (text, tables, extraction_issue) 3-튜플을 반환하도록
        변경. curves > min_curve_signal 이면 'vector_curve_text', images가 있으면
        'scanned_image'로 원인 구분. process_document가 문서 단위로 집계해서
        실패율이 높으면 경고 출력 + 진단 CSV로 저장 (Cell 7 이후 신규 export 셀).
     -> 아직 OCR/LlamaParse 폴백 엔진 자체는 붙이지 않음 (실패율 실측 후 결정 예정,
        보류 목록 참고).
  2. [Cell 3, Cell 7] Chroma-on-Drive 손상 사고 재발 방지. 실제로 인덱싱 중
     `Error loading hnsw index`를 겪었고, 팀원(test01_jiye)도 동일 문제로 아예
     FAISS로 교체한 사례를 확인 — 원인은 Drive FUSE 마운트의 비원자적 쓰기가
     Chroma HNSW의 mmap 파일 구조와 상성이 나쁜 것.
     -> persist_dir을 로컬(/content/chroma_db_local)로 변경. Drive 경로는
        drive_backup_dir로 분리. 세션 시작 시 Drive 백업 -> 로컬 복원,
        10건 처리마다 자동 로컬 -> Drive 백업, 루프 종료 시 최종 백업.
        Cell 7b/Cell R도 동일하게 복원 로직 추가.

v5에서 보류한 것 (다음 논의 필요, 아직 미반영):
  3. OCR(Tesseract) vs LlamaParse(Vision-LLM) 폴백 엔진 선택 -> 위 1번 진단
     로깅으로 5~10개사 지속가능경영보고서 실패율을 먼저 실측한 뒤 결정.
     결론 예정: 사업/지배구조보고서는 폴백 자체가 거의 불필요(텍스트 네이티브
     PDF), 지속가능경영보고서는 표 안에 정량 ESG 지표가 많아 Tesseract보다
     LlamaParse가 유력 (Tesseract는 표 구조를 못 살림).
  4. BM25 토크나이저 품사 필터링 (test01_jiye 제안: NNG/NNP/NNB/VV/VA/SL/SN/SH만
     남기고 조사/어미 제외) -> 비용/리스크 낮아 별도로 진행 가능, 이번엔 범위 밖.

v3 -> v4 수정 사항 (검증 CSV 리뷰 피드백 반영):
  1. [Cell 5c] section_title 키워드 매칭이 순서 없이 "짧은 줄에 키워드 포함"만
     보다가, 연결재무제표 주석의 "1. 회사의 개요"(각주 제목)를 대분류로 다시
     잡아버려 p.57~283이 전부 "회사의 개요"로 오염되는 버그 확인.
     -> detect_section_title이 (제목, 순서인덱스)를 반환하도록 변경하고,
        process_document에서 idx가 역행하면 무시하도록 수정 (섹션은 항상
        문서 순서대로만 진행된다고 가정).
  2. [Cell 6] DART 뷰어 푸터("전자공시시스템 dart.fss.or.kr Page N") 및
     목차 점선(".....") 청크가 그대로 인덱싱되던 문제 확인 (검증 CSV에서
     50자 미만 청크 134개, 그중 다수가 푸터).
     -> strip_noise_lines()로 페이지 텍스트에서 푸터 줄 제거,
        is_toc_chunk()로 목차성 청크는 애초에 append하지 않도록 필터 추가.
        20자 미만 잔여 청크도 함께 스킵.
  3. [Cell 5] 지속가능경영보고서는 선(border) 없는 디자인 표가 많아
     find_tables() 기본 전략(lines)으로 표를 거의 못 잡음 (검증 CSV 기준
     사업/지배구조 대비 페이지당 표 개수가 1/3 이하, 평균 길이도 짧음).
     -> extract_page_content에 doc_type 인자를 추가, sustainability_report이고
        기본 전략으로 표가 0개면 vertical/horizontal_strategy="text"로 재시도.
        호출부([Cell 6], [Cell Q], [Cell Q2]/preview 함수) 전부 doc_type 전달.

v2 -> v3 수정 사항 (GPT 피드백 반영):
  1. Chroma add_documents에 chunk_id를 명시적 ID로 전달 -> 재실행 시 중복 적재 방지
  2. 체크포인트를 {doc_id: {hash, indexed_at}} 구조로 변경 -> PDF가 바뀌면
     자동 감지, 기존 doc_id 청크를 삭제하고 재인덱싱
  3. 큰 표를 token 기준으로 행 단위 분할 (헤더 반복) -> 800토큰 룰이 표에도 적용
  4. 표 셀 클리닝 (줄바꿈/파이프 이스케이프) -> 마크다운 표 깨짐 방지
  5. indexed_at을 문서 단위로 1회만 생성 -> 청크마다 값이 미세하게 달라지는 것 방지
  [가장 중요] 6. BM25가 메타데이터 필터를 못 받던 구조적 문제 해결:
     전체 코퍼스로 BM25를 한 번에 만드는 대신, 검색 시점에 필터(company_code 등)로
     좁혀진 부분집합에서만 BM25를 만드는 FilteredHybridRetriever로 교체.
     -> 이 변경으로 기존 "BM25 전체 재구축 비용" 문제도 자연히 해소되어
        전역 BM25 pickle 저장/로드 로직 자체를 제거함 (Cell 8, Cell R 단순화)

보류한 피드백 (아직 미반영, TODO로 유지):
  - applicable_item_codes 사전 태깅 (현재 빈 배열)
  - esg_area 규칙 기반 태깅 (현재 빈 문자열)
  - 하이브리드 결합 가중치(dense:BM25) 튜닝 — 평가셋 확인 후 결정
  - 리랭커(Cross-Encoder) 적용 여부

확정된 하이퍼파라미터/모델 (팀 확인 완료):
  - 임베딩 모델: Google Gemini gemini-embedding-001 (Google AI Studio API)
  - 벡터스토어: Chroma (로컬, Google Drive에 영속 저장)
  - 청크 크기: 800 tokens / 오버랩 100 tokens (주의: cl100k_base 기준 근사치.
    한국어는 Gemini 실제 토크나이저 기준 청크당 약 450~550토큰 수준일 가능성 높음)
  - BM25 토크나이저: Kiwi (한국어 형태소 분석)
==============================================================================


### [Cell 1] 설치

In [1]:
!pip install -q langchain langchain-google-genai langchain-chroma langchain-community \
    chromadb pdfplumber kiwipiepy rank_bm25 tiktoken tenacity pymupdf

# [v5.2] 검증용 버전 — LlamaParse(3단계)는 이번엔 아예 뺌 (속도 문제).
# 필요해지면 v5.1 파일 참고해서 다시 붙일 것.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 706.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 117.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 51.2 MB/s eta 0:00:00
   ━━━━━━

### [Cell 2] Google Drive 마운트 + API 키

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import getpass
import os

os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google AI Studio API Key 입력: ")
# [v5.2] LlamaParse 키 입력은 이 버전에선 안 씀 (2단계 검증 전용)

Mounted at /content/drive
Google AI Studio API Key 입력: ··········


### [Cell 3] 설정값 (CONFIG)

In [3]:
CONFIG = {
    # --- 확정 ---
    "embedding_model": "gemini-embedding-001",  # Google AI Studio (Gemini Developer API)
    "chunk_size_tokens": 800,
    "chunk_overlap_tokens": 100,
    "vectorstore_backend": "chroma",
    "bm25_tokenizer": "kiwi",
    "retrieval_k": 10,  # dense/BM25 각각 몇 개씩 뽑을지

    # --- 임베딩 API 429(RESOURCE_EXHAUSTED) 대응 ---
    # 확인된 한도: 분당 3,000 요청 / 분당 100만 토큰 / 일일 무제한 (Gemini Embedding)
    # -> 배치를 키우고 딜레이를 줄임. 재시도는 순간 버스트 대비용으로 유지.
    "embed_batch_size": 100,      # 문서당 수백 개 청크를 100개씩 나눠 전송
    "embed_batch_delay_sec": 1.0,  # 배치 사이 짧은 대기 (여전히 안전 마진 목적)
    "embed_retry_max_attempts": 6,  # 429 발생 시 재시도 횟수 (지수 백오프)

    # --- 경로 ---
    "reports_root": "/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/보고서 3종",

    # [v5 수정] Chroma-on-Drive 손상 사고 재발 방지: 인덱싱은 항상 로컬(/content)에서
    # 하고, Drive 경로는 "백업 전용"으로 분리. persist_dir을 Drive에 직접 물리면
    # HNSW 인덱스가 손상될 수 있음 (실제로 겪음 + 팀원도 동일 사고로 FAISS 전환 사례).
    "persist_dir": "/content/chroma_db_local",
    # [중요, 오늘 밤 사고 이후 수정] 원래 chroma_db 경로는 손상 이력이 있고,
    # 검증도 안 된 상태. 마이그레이션으로 복구했고 오늘 LlamaParse 재처리분(68건)까지
    # 반영된 chroma_db_migrated가 지금 가장 최신·검증된 정본이므로 이걸 직접 씀.
    "drive_backup_dir": "/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/인덱스 결과/chroma_db_migrated",
    "backup_every_n_docs": 3,  # 몇 건 처리마다 로컬 -> Drive 자동 백업할지

    "manifest_csv": "/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/인덱스 결과/manifest.csv",
    "checkpoint_log": "/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/인덱스 결과/checkpoint_completed.json",

    # [v5 신규] 텍스트 추출 실패 진단 임계값
    "min_text_threshold": 100,   # 이 미만이면 "텍스트가 거의 없다"로 의심
    "min_curve_signal": 50,      # 페이지 내 curve 오브젝트가 이보다 많으면 벡터 윤곽선 폰트로 판단
    "extraction_diag_csv": "/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/인덱스 결과/extraction_diagnostics.csv",

    # [v5.2] LlamaParse 설정 없음 (이 버전은 2단계 검증 전용)

    # --- TODO: 평가셋 확인 후 결정 ---
    "hybrid_weights": [0.5, 0.5],  # [dense, bm25] 임시값. 튜닝 전까지 변경 금지
    "use_reranker": False,
}

import shutil


def restore_local_from_drive_backup():
    """세션 시작 시 1회 호출. 로컬에 인덱스가 없고 Drive 백업이 있으면 복원.
    (Drive에서 직접 여는 게 아니라, Drive -> 로컬로 통째 복사만 함 — 복사 자체는
    원자적 쓰기가 필요 없어서 안전함.)"""
    import os
    if os.path.exists(CONFIG["persist_dir"]):
        print(f"로컬 인덱스 이미 존재: {CONFIG['persist_dir']} (복원 스킵)")
        return
    if os.path.exists(CONFIG["drive_backup_dir"]):
        print(f"Drive 백업 -> 로컬 복원 중: {CONFIG['drive_backup_dir']}")
        shutil.copytree(CONFIG["drive_backup_dir"], CONFIG["persist_dir"])
        print("복원 완료")
    else:
        print("Drive 백업 없음 — 새 인덱스로 시작")


def backup_local_to_drive():
    """로컬 인덱스를 Drive로 백업 (덮어쓰기). 여러 건 처리 후 주기적으로 호출."""
    import os
    if not os.path.exists(CONFIG["persist_dir"]):
        return
    shutil.copytree(CONFIG["persist_dir"], CONFIG["drive_backup_dir"], dirs_exist_ok=True)

# ==============================================================================
# manifest.csv는 이제 손으로 만들 필요 없음 — 아래 [Cell 4b]가 reports_root
# 폴더를 스캔해서 자동 생성함. 폴더/파일명 규칙:
#   보고서 3종/회사명(기업코드)/기업코드_연도_business.pdf   (business/governance/sustainability)
# 이 규칙을 벗어나는 폴더·파일은 자동으로 건너뛰고 콘솔에 경고 출력됨.
# ==============================================================================

### [Cell 4] 라이브러리 import

In [4]:
import hashlib
import json
import time
from datetime import datetime

import pandas as pd
import pdfplumber
import tiktoken
from kiwipiepy import Kiwi
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_message

kiwi = Kiwi()
RRF_K = 60  # Reciprocal Rank Fusion 표준 상수 (튜닝 대상 아님)

try:
    _enc = tiktoken.encoding_for_model(CONFIG["embedding_model"])
except KeyError:
    _enc = tiktoken.get_encoding("cl100k_base")


def token_len(text: str) -> int:
    """임베딩 모델 기준 토큰 수 근사 계산 (Gemini 전용 tokenizer가 없어 cl100k_base로 근사, 청크 크기 산정용 참고치)."""
    return len(_enc.encode(text))


def kiwi_tokenize(text: str):
    """BM25용 한국어 형태소 분석 토크나이저 (조사 분리 목적)."""
    return [t.form for t in kiwi.tokenize(text)]


splitter = RecursiveCharacterTextSplitter(
    chunk_size=CONFIG["chunk_size_tokens"],
    chunk_overlap=CONFIG["chunk_overlap_tokens"],
    length_function=token_len,
    separators=["\n\n", "\n", ". ", " ", ""],
)


@retry(
    wait=wait_exponential(multiplier=2, min=5, max=90),
    stop=stop_after_attempt(CONFIG["embed_retry_max_attempts"]),
    retry=retry_if_exception_message(match=".*RESOURCE_EXHAUSTED.*|.*429.*"),
    reraise=True,
)
def _add_batch_with_retry(vectorstore, batch_chunks):
    ids = [c.metadata["chunk_id"] for c in batch_chunks]
    vectorstore.add_documents(batch_chunks, ids=ids)


def add_documents_throttled(vectorstore, chunks, batch_size, delay_sec):
    """청크를 작은 배치로 나눠서 넣고, 배치 사이에 대기 -> 429(RESOURCE_EXHAUSTED) 완화.
    배치 하나가 재시도까지 다 실패하면 그 배치만 실패 처리하고 나머지는 계속 진행."""
    total = len(chunks)
    for i in range(0, total, batch_size):
        batch = chunks[i:i + batch_size]
        try:
            _add_batch_with_retry(vectorstore, batch)
        except Exception as e:
            print(f"  배치 실패 (청크 {i}~{i + len(batch)}/{total}): {e}")
            raise  # 문서 단위 실패로 취급 -> Cell 7의 except에서 처리
        if i + batch_size < total:
            time.sleep(delay_sec)

/tmp/ipykernel_541/1235179789.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


### [Cell 4b] manifest.csv 자동 생성 (폴더 스캔)

In [5]:
# 폴더/파일명이 규칙적이라 손으로 100줄 안 쳐도 됨.
#   기대 구조: {reports_root}/회사명(기업코드)/기업코드_연도_business.pdf
#   doc_type 파일명 표기(business/governance/sustainability)를 내부 스키마
#   값(business_report 등)으로 매핑. 규칙에 안 맞는 폴더·파일은 건너뛰고 경고만 출력.
import re
from pathlib import Path

DOC_TYPE_MAP = {
    "business": "business_report",
    "governance": "governance_report",
    "sustainability": "sustainability_report",
}
FOLDER_PATTERN = re.compile(r"^(.*)\((\d{6})\)$")
FILE_PATTERN = re.compile(r"^(\d{6})_(\d{4})_(business|governance|sustainability)\.pdf$")


def build_manifest(reports_root: str) -> pd.DataFrame:
    root = Path(reports_root)
    if not root.exists():
        raise FileNotFoundError(f"경로를 찾을 수 없음: {reports_root} (Drive 마운트/폴더명 확인)")

    rows = []
    for company_dir in sorted(root.iterdir()):
        if not company_dir.is_dir():
            continue
        m = FOLDER_PATTERN.match(company_dir.name)
        if not m:
            print(f"[건너뜀] 폴더명 규칙 불일치: {company_dir.name}")
            continue
        company_name, company_code = m.group(1), m.group(2)

        for pdf_file in sorted(company_dir.glob("*.pdf")):
            fm = FILE_PATTERN.match(pdf_file.name)
            if not fm:
                print(f"[건너뜀] 파일명 규칙 불일치: {pdf_file.name}")
                continue
            file_code, year, doc_type_short = fm.groups()
            if file_code != company_code:
                print(f"[경고] 폴더코드({company_code})와 파일코드({file_code}) 불일치: {pdf_file}")
                continue
            rows.append({
                "company_code": company_code,
                "company_name": company_name,
                "year": int(year),
                "doc_type": DOC_TYPE_MAP[doc_type_short],
                "file_path": str(pdf_file),
            })

    return pd.DataFrame(rows)


manifest_df = build_manifest(CONFIG["reports_root"])

# [v6] 테스트 필터 제거 — 전체 회사 자동 인식.
# 위 build_manifest()의 [건너뜀]/[경고] 로그를 먼저 확인할 것 (파일명 규칙
# 안 맞으면 조용히 스킵되므로, 아래 검증으로 누락을 바로 잡아낼 것).
print(f"인식된 문서: {len(manifest_df)}건")
print(f"인식된 회사: {manifest_df['company_code'].nunique()}개")

# 기대 회사 수와 비교 (실제 목표 회사 수로 수정할 것)
EXPECTED_COMPANY_COUNT = 100
missing = EXPECTED_COMPANY_COUNT - manifest_df["company_code"].nunique()
if missing > 0:
    print(f"⚠️ {missing}개사가 누락됐을 수 있음 — 위 [건너뜀]/[경고] 로그 확인할 것")

# 회사별 문서 개수가 3(문서유형)*N(연도) 미만인 곳 — 일부 문서만 있거나 파일명 오류 가능성
counts = manifest_df.groupby("company_code").size()
print("\n문서 개수가 적은 회사 (부족한 연도/문서종류 없는지 확인할 것):")
print(counts.sort_values().head(10))

os.makedirs(os.path.dirname(CONFIG["manifest_csv"]), exist_ok=True)
manifest_df.to_csv(CONFIG["manifest_csv"], index=False, encoding="utf-8-sig")
print(f"\nmanifest 저장 완료: {len(manifest_df)}건")
manifest_df

[건너뜀] 파일명 규칙 불일치: 004710_2023_sustainability_hansoltechnics_only.pdf
[건너뜀] 파일명 규칙 불일치: 004710_2024_sustainability_hansoltechnics_only.pdf
[건너뜀] 파일명 규칙 불일치: 004710_2025_sustainability_hansoltechnics_only.pdf
인식된 문서: 450건
인식된 회사: 54개
⚠️ 46개사가 누락됐을 수 있음 — 위 [건너뜀]/[경고] 로그 확인할 것

문서 개수가 적은 회사 (부족한 연도/문서종류 없는지 확인할 것):
company_code
019180    2
189300    2
393890    2
450080    3
033240    4
005680    4
060720    4
011930    5
097520    5
033640    5
dtype: int64

manifest 저장 완료: 450건


,company_code,company_name,year,doc_type,file_path
0,000990,DB하이텍,2022,business_report,/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/...
1,000990,DB하이텍,2022,governance_report,/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/...
2,000990,DB하이텍,2022,sustainability_report,/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/...
3,000990,DB하이텍,2023,business_report,/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/...
4,000990,DB하이텍,2023,governance_report,/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/...
...,...,...,...,...,...
445,298040,효성중공업,2024,governance_report,/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/...
446,298040,효성중공업,2024,sustainability_report,/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/...
447,298040,효성중공업,2025,business_report,/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/...
448,298040,효성중공업,2025,governance_report,/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/...


### [Cell 4c] (대안) 내 드라이브에 안 넣고, 공유 링크의 폴더 ID로 직접 접근

In [ ]:
# reports_root가 진짜 "공유 드라이브"(Shareddrives)가 아니라 그냥 "공유된 폴더"라서
# drive.mount()로 안 보일 때 이 셀을 Cell 4b 대신 사용.
# 사용법: 공유 링크 https://drive.google.com/drive/folders/<FOLDER_ID> 에서
#         <FOLDER_ID> 부분만 복사해서 아래 SHARED_FOLDER_ID에 붙여넣기.
#         (뷰어 권한만 있어도 동작함 — 내 드라이브에 추가할 필요 없음)
from google.colab import auth
import google.auth
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload
import io

SHARED_FOLDER_ID = "여기에_폴더ID_붙여넣기"
LOCAL_CACHE_DIR = "/content/reports_cache"  # 코랩 로컬 디스크 (세션 종료 시 삭제됨, 재실행 시 재다운로드)

auth.authenticate_user()
_creds, _ = google.auth.default()
drive_service = gbuild("drive", "v3", credentials=_creds)


def _list_children(folder_id: str) -> list[dict]:
    items, page_token = [], None
    while True:
        resp = drive_service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken, files(id, name, mimeType)",
            pageToken=page_token,
        ).execute()
        items.extend(resp.get("files", []))
        page_token = resp.get("nextPageToken")
        if not page_token:
            break
    return items


def _download_file(file_id: str, dest_path: str):
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    request = drive_service.files().get_media(fileId=file_id)
    with io.FileIO(dest_path, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()


def build_manifest_from_drive_api(root_folder_id: str, local_cache_dir: str) -> pd.DataFrame:
    rows = []
    company_folders = [
        f for f in _list_children(root_folder_id)
        if f["mimeType"] == "application/vnd.google-apps.folder"
    ]
    for folder in company_folders:
        m = FOLDER_PATTERN.match(folder["name"])
        if not m:
            print(f"[건너뜀] 폴더명 규칙 불일치: {folder['name']}")
            continue
        company_name, company_code = m.group(1), m.group(2)

        pdf_files = [f for f in _list_children(folder["id"]) if f["name"].lower().endswith(".pdf")]
        for pdf in pdf_files:
            fm = FILE_PATTERN.match(pdf["name"])
            if not fm:
                print(f"[건너뜀] 파일명 규칙 불일치: {pdf['name']}")
                continue
            file_code, year, doc_type_short = fm.groups()
            if file_code != company_code:
                print(f"[경고] 폴더코드({company_code})와 파일코드({file_code}) 불일치: {pdf['name']}")
                continue

            local_path = os.path.join(local_cache_dir, folder["name"], pdf["name"])
            if not os.path.exists(local_path):
                print(f"다운로드 중: {folder['name']}/{pdf['name']}")
                _download_file(pdf["id"], local_path)

            rows.append({
                "company_code": company_code,
                "company_name": company_name,
                "year": int(year),
                "doc_type": DOC_TYPE_MAP[doc_type_short],
                "file_path": local_path,
            })
    return pd.DataFrame(rows)


# 사용 예 (Cell 4b의 build_manifest() 대신 이걸로 manifest_df 생성):
# manifest_df = build_manifest_from_drive_api(SHARED_FOLDER_ID, LOCAL_CACHE_DIR)
# os.makedirs(os.path.dirname(CONFIG["manifest_csv"]), exist_ok=True)
# manifest_df.to_csv(CONFIG["manifest_csv"], index=False, encoding="utf-8-sig")
# print(f"manifest 생성 완료: {len(manifest_df)}건")
# manifest_df

### [Cell 5] 표/텍스트 추출 유틸

In [6]:
def clean_cell(cell) -> str:
    """표 셀 안의 줄바꿈/파이프가 마크다운 표 구조를 깨는 것 방지."""
    if cell is None:
        return ""
    return str(cell).replace("\n", " ").replace("|", "\\|").strip()


def chunk_table(table_data: list[list], max_tokens: int) -> list[dict]:
    """큰 표를 token 기준으로 행 단위 분할 (각 조각에 헤더 반복).
    반환: [{"markdown": "...", "rows": [[...], ...]}, ...]
    """
    header = [clean_cell(c) for c in table_data[0]]
    data_rows = table_data[1:]

    header_md = (
        "| " + " | ".join(header) + " |\n"
        + "|" + "|".join(["---"] * len(header)) + "|"
    )

    def render(rows):
        body = "\n".join(
            "| " + " | ".join(clean_cell(c) for c in r) + " |" for r in rows
        )
        return header_md + ("\n" + body if body else "")

    chunks = []
    current_rows: list[list] = []
    for row in data_rows:
        candidate = current_rows + [row]
        if token_len(render(candidate)) > max_tokens and current_rows:
            chunks.append({"markdown": render(current_rows), "rows": current_rows})
            current_rows = [row]
        else:
            current_rows = candidate
    if current_rows:
        chunks.append({"markdown": render(current_rows), "rows": current_rows})

    return chunks or [{"markdown": header_md, "rows": []}]


# [v4 패치] doc_type 인자 추가: 지속가능경영보고서는 선(border) 없는 디자인
# 표가 많아 기본 lines 전략으로 표를 거의 못 잡는 문제 확인 (검증 CSV에서
# 사업/지배구조 대비 페이지당 표 개수 1/3 이하). 기본 전략으로 0개면
# text 전략(좌표 기반 텍스트 정렬)으로 한 번 더 시도.
# [v5 신규] 텍스트 추출이 "조용히" 실패하는 걸 막기 위한 진단.
# jin(DB하이텍) 케이스: 본문이 폰트가 아니라 벡터 윤곽선(curve)으로 변환된 PDF라
# pdfplumber로 92p 중 87p(95%)에서 텍스트가 전혀 안 뽑혔는데, v4는 이를 감지하지
# 못하고 빈 문자열로 조용히 넘어갔음. curves/images 개수로 원인을 구분해서 진단만
# 하고, 실제 OCR/LlamaParse 폴백 엔진은 아직 안 붙임 (실패율 실측 후 결정).
def extract_page_content(pdf_page, doc_type: str = ""):
    """표 영역을 찾고, 그 바운딩박스를 제외한 텍스트만 반환 (표-텍스트 중복 방지).
    주의(GPT 지적): PDF 레이아웃에 따라 bbox가 완전히 안 잡힐 수 있음.
    실제 대상 PDF 몇 개는 아래 Cell Q로 반드시 샘플 확인할 것.

    반환값 3번째 요소(extraction_issue)는 [v5 신규]:
      None                 -> 정상 (텍스트 충분 / 혹은 표지·간지처럼 원래 내용이 적음)
      "vector_curve_text"  -> 텍스트 거의 없는데 curve 오브젝트는 많음 (벡터 윤곽선 폰트 의심)
      "scanned_image"      -> 텍스트 거의 없는데 이미지 오브젝트 있음 (스캔본 의심)

    반환값 4번째 요소(table_bboxes)는 [v5.1 신규]: pdfplumber가 찾은 표 영역 bbox 리스트.
    PyMuPDF/LlamaParse로 재추출할 때 표 영역과 겹치는 부분을 제외해서 표-텍스트
    중복 적재(v1급 버그)를 막는 데 재사용됨.
    """
    tables = pdf_page.find_tables()

    # [v4 패치] 지속가능경영보고서에서 기본(lines) 전략으로 표가 하나도
    # 안 잡히면 text 전략으로 재시도 (선 없는 디자인 표 대응)
    if doc_type == "sustainability_report" and not tables:
        tables = pdf_page.find_tables(
            table_settings={"vertical_strategy": "text", "horizontal_strategy": "text"}
        )

    table_bboxes = [t.bbox for t in tables]

    table_data_list = []
    for t in tables:
        try:
            extracted = t.extract()
        except Exception:
            continue
        if extracted and len(extracted) >= 2:
            table_data_list.append(extracted)

    def not_in_table(obj):
        x0, top, x1, bottom = obj.get("x0"), obj.get("top"), obj.get("x1"), obj.get("bottom")
        if x0 is None:
            return True
        for (tx0, ttop, tx1, tbottom) in table_bboxes:
            if x0 >= tx0 - 1 and x1 <= tx1 + 1 and top >= ttop - 1 and bottom <= tbottom + 1:
                return False
        return True

    non_table_page = pdf_page.filter(not_in_table)
    text = non_table_page.extract_text() or ""

    # [v5 신규] 실패 진단 (표지/간지처럼 원래 텍스트가 적은 정상 페이지와 구분)
    extraction_issue = None
    if len(text.strip()) < CONFIG.get("min_text_threshold", 100):
        try:
            n_curves = len(pdf_page.curves)
            n_images = len(pdf_page.images)
        except Exception:
            n_curves, n_images = 0, 0
        if n_curves > CONFIG.get("min_curve_signal", 50):
            extraction_issue = "vector_curve_text"
        elif n_images > 0:
            extraction_issue = "scanned_image"
        # else: curve/image 신호도 없으면 진짜로 내용이 적은 페이지(표지 등) -> None 유지

    return text, table_data_list, extraction_issue, table_bboxes  # [v5.1] table_bboxes 추가


def base_metadata(company_code, company_name, year, doc_type, doc_id,
                   chunk_id, page, content_type, source_file_hash,
                   doc_indexed_at, table_rows=None, section_title=""):
    """Chroma는 메타데이터로 str/int/float/bool만 허용 (None, list 불가)
    -> None은 빈 문자열로, list는 JSON 문자열로 변환."""
    return {
        "company_code": company_code,
        "company_name": company_name,
        "year": year,
        "doc_type": doc_type,
        "doc_id": doc_id,
        "chunk_id": chunk_id,
        "page": page,
        # 사업보고서/지배구조보고서: Cell 5c 키워드 탐지로 채워짐
        # 지속가능경영보고서: 목차 규칙이 없어 아직 빈 문자열 (팀 결정: 평가셋 이후 보강)
        "section_title": section_title,
        "content_type": content_type,
        "table_data": json.dumps(table_rows, ensure_ascii=False) if table_rows else "",
        "esg_area": "",  # TODO: 필요 시 규칙 기반 태깅
        "applicable_item_codes": json.dumps([], ensure_ascii=False),  # TODO
        "embedding_model": CONFIG["embedding_model"],
        "indexed_at": doc_indexed_at,  # 문서 단위 1회 생성값 재사용
        "source_file_hash": source_file_hash,
    }


# [v5.1 신규] PyMuPDF 캐스케이딩 — pdfplumber가 실패로 진단한 페이지만 재시도.
# 10개사 실측 결과: DB하이텍(진짜 벡터 윤곽선 폰트)은 여전히 실패했지만,
# 경동나비엔(97% 복구)·가온전선(91% 복구)처럼 "pdfplumber만 못 읽었을 뿐" 텍스트가
# 실제로 있던 케이스는 대부분 무료로 복구됨.
#
# 주의: PyMuPDF의 page.get_text("blocks")는 표 영역도 그대로 포함해서 뽑아오기
# 때문에, pdfplumber가 이미 찾아둔 table_bboxes와 겹치는 블록은 제외해야 함.
# 안 그러면 같은 표 내용이 마크다운 표 청크(pdfplumber) + 뒤죽박죽 텍스트 청크
# (PyMuPDF)로 이중 적재되는 v1급 버그가 재발함.
#
# 좌표계 주의: pdfplumber와 PyMuPDF 모두 페이지 좌상단 원점 기준 좌표를 쓰지만,
# 실제 파일에서 페이지 회전(rotate) 값이 있으면 어긋날 수 있음. 처음 몇 건은
# 반드시 Cell Q로 표 중복 여부를 눈으로 확인할 것.
def pymupdf_rescue_text(fitz_page, table_bboxes, margin: float = 1.0) -> str:
    """PyMuPDF로 페이지를 재추출하되, table_bboxes와 겹치는 블록은 제외."""
    blocks = fitz_page.get_text("blocks")  # [(x0, y0, x1, y1, text, block_no, block_type), ...]

    def in_any_table(x0, y0, x1, y1):
        for (tx0, ttop, tx1, tbottom) in table_bboxes:
            if (x0 >= tx0 - margin and x1 <= tx1 + margin
                    and y0 >= ttop - margin and y1 <= tbottom + margin):
                return True
        return False

    kept = [
        b[4] for b in blocks
        if not in_any_table(b[0], b[1], b[2], b[3])
    ]
    return "\n".join(kept)



### [Cell 5c] 키워드 기반 section_title 탐지 (사업보고서·지배구조보고서)

In [7]:
#
# 주의: 아래 키워드는 일반적인 법정 서식 기준의 "출발점"일 뿐, 회사·연도별로
# 소제목 문구나 넘버링 방식(로마자/아라비아숫자/장 표기 등)이 다를 수 있음.
# 정확한 문구 매칭 대신 "짧은 줄 안에 핵심 키워드가 포함되는지"만 보는 방식이라,
# 넘버링 형식이 달라도 잡히고, 매칭 실패 시 빈 문자열로 안전하게 degrade됨.
# 반드시 Cell Q2로 5~10개사 검증 후 키워드 목록을 조정할 것.
#
# 페이지 단위 granularity: 한 페이지 안에서 섹션이 바뀌면 그 페이지의 마지막
# 매칭을 채택 (청크 단위로 정밀하게 자르지는 않음 — 알려진 한계).
#
# [v4 패치] 리스트 순서 = 문서상 등장 순서라고 가정하고, 키워드 인덱스를 같이
# 반환하도록 변경. 예: 연결재무제표 주석의 "1. 회사의 개요"(각주 제목)처럼
# 이미 지나간 대분류 키워드가 뒤에서 다시 매칭되는 경우, 호출부(process_document)
# 에서 인덱스 역행을 감지해 무시한다. (검증 CSV에서 이 문제로 p.57~283이
# 전부 "회사의 개요"로 오염된 것 확인 — 재무제표 주석 안의 각주 제목이 원인)
SECTION_KEYWORDS = {
    "business_report": [
        "회사의 개요",
        "사업의 내용",
        "재무에 관한 사항",
        "이사의 경영진단",
        "회계감사인의 감사의견",
        "이사회 등 회사의 기관에 관한 사항",
        "주주에 관한 사항",
        "임원 및 직원 등에 관한 사항",
        "계열회사 등에 관한 사항",
        "이해관계자와의 거래내용",
    ],
    "governance_report": [
        "주주",
        "이사회",
        "감사기구",
        "이사 감사의 보수",
        "내부통제",
        "배당",
        "주주총회",
        "공시",
    ],
}


def _normalize(line: str) -> str:
    return line.replace("·", "").replace(" ", "").replace("/", "")


def detect_section_title(line: str, doc_type: str) -> tuple[str, int] | None:
    """짧고 키워드를 포함한 줄을 섹션 제목 후보로 판단.
    확신 없으면 None 반환 (호출부에서 빈 문자열로 처리 -> 안전).

    [v4 패치] 반환 타입을 str -> tuple[str, int]로 변경. int는
    SECTION_KEYWORDS 리스트 상의 순서 인덱스이며, 호출부에서 역행 판단에 사용.
    """
    keywords = SECTION_KEYWORDS.get(doc_type)
    if not keywords:
        return None

    stripped = line.strip()
    if not stripped or len(stripped) > 40:  # 제목치고 너무 길면 본문일 가능성 높음
        return None

    norm_line = _normalize(stripped)
    for idx, kw in enumerate(keywords):
        if _normalize(kw) in norm_line:
            return (kw, idx)
    return None


### [Cell 6] 문서 1건(PDF 1개) 처리 → Document 리스트 생성

In [8]:
# [v4 패치] 아래 두 헬퍼를 process_document보다 먼저 정의:
#   - strip_noise_lines: DART 뷰어 푸터("전자공시시스템 dart.fss.or.kr Page N")를
#     페이지 텍스트에서 제거. 검증 CSV에서 50자 미만 청크 134개 중 다수가 이 푸터였음.
#   - is_toc_chunk: 점선(".....")이 많이 포함된 목차성 청크 판별.
#     검증 CSV에서 목차 청크가 최대 5,388자까지 그대로 인덱싱되어 있었음.
#
# [v5 신규] extraction_diagnostics: 문서 단위 텍스트 추출 실패 진단 결과를 쌓아두는
# 전역 리스트. process_document가 호출될 때마다 이어서 누적되고, Cell 7 루프가 끝난
# 뒤 별도 셀에서 CSV로 export함 (OCR/LlamaParse 폴백 도입 여부 판단용 실측 데이터).
import re

import fitz  # PyMuPDF, [v5.1 신규] 캐스케이딩용

extraction_diagnostics: list[dict] = []

NOISE_LINE_PATTERN = re.compile(r"^전자공시시스템\s*dart\.fss\.or\.kr\s*Page\s*\d+\s*$")
TOC_DOT_PATTERN = re.compile(r"\.{10,}")


def strip_noise_lines(text: str) -> str:
    """DART 뷰어 푸터 줄 제거."""
    return "\n".join(
        ln for ln in text.split("\n") if not NOISE_LINE_PATTERN.match(ln.strip())
    )


def is_toc_chunk(chunk_text: str) -> bool:
    """점선 목차(.......) 덩어리 판별 (점 10개 이상 연속이 3군데 이상)."""
    return len(TOC_DOT_PATTERN.findall(chunk_text)) >= 3


def process_document(company_code: str, company_name: str, year: int,
                      doc_type: str, file_path: str,
                      source_file_hash: str, doc_indexed_at: str) -> list[Document]:

    doc_id = f"{company_code}_{year}_{doc_type}"
    chunks: list[Document] = []
    counter = 0
    current_section = ""      # 사업/지배구조보고서만 Cell 5c로 갱신됨 (페이지 단위 granularity)
    current_section_idx = -1  # [v4 패치] SECTION_KEYWORDS 상의 순서 인덱스, 역행 감지용
    doc_issue_pages: list[dict] = []      # [v5] 최초(pdfplumber) 실패로 진단된 페이지들
    doc_rescued_pymupdf = 0               # [v5.1 신규] PyMuPDF로 복구된 페이지 수
    doc_rescued_llamaparse = 0            # [v5.1 신규] LlamaParse로 복구된 페이지 수 (비활성 시 항상 0)
    doc_still_failed: list[dict] = []     # [v5.1 신규] 캐스케이딩 다 거치고도 실패한 페이지들

    # [v5.1 신규] PyMuPDF는 pdfplumber와 별도로 같은 파일을 다시 열어야 함
    # (라이브러리가 달라서 문서 핸들을 공유할 수 없음)
    fitz_doc = fitz.open(file_path)

    with pdfplumber.open(file_path) as pdf:
        total_pages = len(pdf.pages)
        for page_number, page in enumerate(pdf.pages):
            # [v4 패치] doc_type 전달 -> 지속가능경영보고서는 표 fallback 적용됨
            # [v5.1 패치] 4-튜플 반환 (table_bboxes 추가 — PyMuPDF 캐스케이딩에 재사용)
            text, tables, extraction_issue, table_bboxes = extract_page_content(page, doc_type)
            text = strip_noise_lines(text)  # [v4 패치] 푸터 줄 제거

            if extraction_issue:  # [v5] 실패 페이지 기록
                doc_issue_pages.append({"page": page_number + 1, "issue": extraction_issue})

                # [v5.1 신규] 2단계: PyMuPDF 재시도 (무료, 표 영역 제외 처리 포함)
                rescued_text = pymupdf_rescue_text(fitz_doc[page_number], table_bboxes)
                rescued_text = strip_noise_lines(rescued_text)
                if len(rescued_text.strip()) >= CONFIG.get("min_text_threshold", 100):
                    text = rescued_text
                    extraction_issue = None
                    doc_rescued_pymupdf += 1
                else:
                    # [v5.2] 이 버전은 2단계(PyMuPDF)까지만. LlamaParse는 별도 검증 후 결정.
                    doc_still_failed.append({"page": page_number + 1, "issue": extraction_issue})

            # 이 페이지에서 새 섹션 제목이 감지되면 갱신 (지속가능경영보고서는
            # SECTION_KEYWORDS에 키가 없어 detect_section_title이 항상 None 반환 -> 그대로 "")
            # [v4 패치] idx가 역행하면(이미 지나간 대분류가 다시 매칭되면) 무시.
            # 재무제표 주석의 "1. 회사의 개요" 같은 각주 제목 오탐 방지.
            for line in text.split("\n"):
                detected = detect_section_title(line, doc_type)
                if not detected:
                    continue
                kw, idx = detected
                if idx >= current_section_idx:
                    current_section = kw
                    current_section_idx = idx

            # --- 표 청크 (큰 표는 토큰 기준으로 여러 조각으로 분할) ---
            for table_data in tables:
                for piece in chunk_table(table_data, CONFIG["chunk_size_tokens"]):
                    counter += 1
                    chunk_id = f"{doc_id}_{counter:04d}"
                    meta = base_metadata(
                        company_code, company_name, year, doc_type, doc_id,
                        chunk_id, page_number + 1, "table", source_file_hash,
                        doc_indexed_at, table_rows=piece["rows"],
                        section_title=current_section,
                    )
                    meta["chunk_char_count"] = len(piece["markdown"])
                    chunks.append(Document(page_content=piece["markdown"], metadata=meta))

            # --- 텍스트 청크 (표 영역 제외된 텍스트만) ---
            if not text.strip():
                continue
            for chunk_text in splitter.split_text(text):
                # [v4 패치] 목차 덩어리 / 20자 미만 잔여 청크는 인덱싱하지 않음
                if is_toc_chunk(chunk_text) or len(chunk_text.strip()) < 20:
                    continue
                counter += 1
                chunk_id = f"{doc_id}_{counter:04d}"
                meta = base_metadata(
                    company_code, company_name, year, doc_type, doc_id,
                    chunk_id, page_number + 1, "text", source_file_hash,
                    doc_indexed_at, section_title=current_section,
                )
                meta["chunk_char_count"] = len(chunk_text)
                chunks.append(Document(page_content=chunk_text, metadata=meta))

    fitz_doc.close()  # [v5.1 신규]

    # [v5.1 수정] 문서 단위 추출 실패율 집계 — 캐스케이딩 전/후를 구분해서 기록.
    # (before = pdfplumber만 썼을 때 실패율, after = PyMuPDF/LlamaParse까지 거친 뒤 잔여 실패율)
    n_before = len(doc_issue_pages)
    n_after = len(doc_still_failed)
    ratio_before = n_before / total_pages if total_pages else 0.0
    ratio_after = n_after / total_pages if total_pages else 0.0
    if n_before > 0:
        issue_types_before = {}
        for rec in doc_issue_pages:
            issue_types_before[rec["issue"]] = issue_types_before.get(rec["issue"], 0) + 1
        print(f"  ⚠️ pdfplumber 실패: {n_before}/{total_pages} ({ratio_before:.0%}) — {issue_types_before}")
        print(f"     -> PyMuPDF 복구: {doc_rescued_pymupdf}건 / "
              f"LlamaParse 복구: {doc_rescued_llamaparse}건 / "
              f"잔여 실패: {n_after}건 ({ratio_after:.0%})")
        if ratio_after > 0.5:
            print(f"     -> 캐스케이딩 후에도 절반 이상 실패. LlamaParse 3단계가 꺼져있다면 "
                  f"켜는 걸 검토할 것: {doc_id}")
    extraction_diagnostics.append({
        "doc_id": doc_id, "company_code": company_code, "company_name": company_name,
        "year": year, "doc_type": doc_type, "file_path": file_path,  # [v5.2] 재처리 스크립트용
        "total_pages": total_pages,
        "issue_pages_before": n_before, "issue_ratio_before": round(ratio_before, 3),
        "rescued_pymupdf": doc_rescued_pymupdf, "rescued_llamaparse": doc_rescued_llamaparse,
        "issue_pages_after": n_after, "issue_ratio_after": round(ratio_after, 3),
        "issue_breakdown_after": json.dumps(
            {k: sum(1 for r in doc_still_failed if r["issue"] == k)
             for k in {r["issue"] for r in doc_still_failed}},
            ensure_ascii=False,
        ),
        # [v5.2 신규] 정확히 몇 페이지가 실패했는지 (1-indexed) — 별도 LlamaParse
        # 재처리 스크립트가 이 목록으로 target_pages를 구성함.
        "still_failed_pages": json.dumps(
            sorted(r["page"] for r in doc_still_failed), ensure_ascii=False
        ),
    })

    return chunks


### [Cell 7] manifest 순회 → 해시 기반 체크포인트 + 변경분만 재인덱싱

In [21]:
# [v5 신규] 세션 시작 시 1회: 로컬에 인덱스 없으면 Drive 백업에서 복원
restore_local_from_drive_backup()

embeddings = GoogleGenerativeAIEmbeddings(model=CONFIG["embedding_model"])
vectorstore = Chroma(
    persist_directory=CONFIG["persist_dir"],  # [v5] 이제 로컬 경로
    embedding_function=embeddings,
)

if os.path.exists(CONFIG["checkpoint_log"]):
    with open(CONFIG["checkpoint_log"], encoding="utf-8") as f:
        checkpoint = json.load(f)  # {doc_id: {"source_file_hash":..., "indexed_at":...}}
else:
    checkpoint = {}

# [v6.1 신규] 정합성 자동 확인: checkpoint엔 완료로 찍혀있는데 실제 로컬 Chroma엔
# 데이터가 없는 문서를 찾아서 checkpoint에서 제거 (백업 주기가 checkpoint 저장
# 주기보다 느려서 생기는 간극 방지 — 세션이 끊겼을 때 실제로 겪었던 문제).
# 이미 정상적으로 들어있는 문서는 이 체크를 통과하고 그대로 스킵되므로
# 절대 재처리되거나 삭제되지 않음.
_missing_from_vectorstore = []
for doc_id in list(checkpoint.keys()):
    _existing = vectorstore.get(where={"doc_id": doc_id}, limit=1)
    if not _existing["ids"]:
        _missing_from_vectorstore.append(doc_id)

if _missing_from_vectorstore:
    print(f"⚠️ checkpoint엔 있지만 실제 데이터가 없는 문서 {len(_missing_from_vectorstore)}건 발견 → 재처리 대상으로 되돌림:")
    for doc_id in _missing_from_vectorstore:
        print(f"   {doc_id}")
        checkpoint.pop(doc_id)
    with open(CONFIG["checkpoint_log"], "w", encoding="utf-8") as f:
        json.dump(checkpoint, f, ensure_ascii=False, indent=2)
else:
    print("정합성 확인 완료: checkpoint와 실제 데이터 일치")

manifest = pd.read_csv(CONFIG["manifest_csv"], dtype={"company_code": str})
_docs_done_since_backup = 0  # [v5 신규]

for _, row in manifest.iterrows():
    doc_id = f"{row['company_code']}_{row['year']}_{row['doc_type']}"
    file_path = row["file_path"]

    # [v6.2 신규] 전체 파일을 읽어 md5 해시부터 내지 않고, 먼저 크기·수정시각
    # (os.stat, 파일 내용을 안 읽으므로 거의 즉시 끝남)만 비교해서 빠르게 스킵 판단.
    # Drive에서 매번 수백 개 PDF 전체를 다시 읽는 게 재시작 후 느려지는 진짜 원인이었음.
    # stat이 이전과 완전히 같으면 "안 바뀐 걸로 간주"하고 해시 계산 자체를 생략.
    # 실제로 파일이 바뀐 경우엔 stat도 거의 항상 같이 바뀌므로 안전함.
    _stat = os.stat(file_path)
    current_stat = {"size": _stat.st_size, "mtime": _stat.st_mtime}

    prior = checkpoint.get(doc_id)
    if prior and prior.get("source_file_stat") == current_stat:
        print(f"스킵 (변경 없음, stat 일치): {doc_id}")
        continue

    # stat이 없거나(v6.1 이전 체크포인트) 다르면 그때만 진짜로 파일을 읽어서 해시 확인
    with open(file_path, "rb") as f:
        current_hash = hashlib.md5(f.read()).hexdigest()

    if prior and prior.get("source_file_hash") == current_hash:
        # 내용은 같은데 stat만 달라진 경우(예: Drive 재동기화로 mtime만 바뀜)
        # -> 재인덱싱 없이 stat 정보만 최신화
        print(f"스킵 (내용 동일, stat만 갱신): {doc_id}")
        checkpoint[doc_id]["source_file_stat"] = current_stat
        with open(CONFIG["checkpoint_log"], "w", encoding="utf-8") as f:
            json.dump(checkpoint, f, ensure_ascii=False, indent=2)
        continue

    if prior:
        print(f"PDF 변경 감지 → 기존 청크 삭제 후 재인덱싱: {doc_id}")
        vectorstore.delete(where={"doc_id": doc_id})

    print(f"처리 중: {doc_id}")
    try:
        doc_indexed_at = datetime.now().isoformat()
        chunks = process_document(
            company_code=row["company_code"],
            company_name=row["company_name"],
            year=int(row["year"]),
            doc_type=row["doc_type"],
            file_path=file_path,
            source_file_hash=current_hash,
            doc_indexed_at=doc_indexed_at,
        )
        if chunks:
            print(f"  청크 {len(chunks)}개, {CONFIG['embed_batch_size']}개씩 배치 임베딩 중...")
            add_documents_throttled(
                vectorstore, chunks,
                batch_size=CONFIG["embed_batch_size"],
                delay_sec=CONFIG["embed_batch_delay_sec"],
            )

        checkpoint[doc_id] = {
            "source_file_hash": current_hash,
            "source_file_stat": current_stat,  # [v6.2 신규] 다음 실행 시 빠른 스킵용
            "indexed_at": doc_indexed_at,
        }
        with open(CONFIG["checkpoint_log"], "w", encoding="utf-8") as f:
            json.dump(checkpoint, f, ensure_ascii=False, indent=2)

        # [v6.1 신규] 주기적 자동 백업
        _docs_done_since_backup += 1
        if _docs_done_since_backup >= CONFIG["backup_every_n_docs"]:
            print(f"  💾 {_docs_done_since_backup}건 완료 → Drive로 자동 백업 중...")
            backup_local_to_drive()
            print("  💾 Drive 자동 백업 완료")
            _docs_done_since_backup = 0

    except Exception as e:
        print(f"실패 (건너뜀): {doc_id} — {e}")
        continue

print(f"\n완료: {len(checkpoint)}건 / 전체 {len(manifest)}건")

# [v6.1 신규] 최종 백업 — 마지막 몇 건이 backup_every_n_docs 문턱을 못 채운 채
# 끝나도 누락되지 않도록
print("최종 Drive 백업 중...")
backup_local_to_drive()
print("최종 백업 완료")

로컬 인덱스 이미 존재: /content/chroma_db_local (복원 스킵)
정합성 확인 완료: checkpoint와 실제 데이터 일치
스킵 (변경 없음, stat 일치): 000990_2022_business_report
스킵 (변경 없음, stat 일치): 000990_2022_governance_report
스킵 (변경 없음, stat 일치): 000990_2022_sustainability_report
스킵 (변경 없음, stat 일치): 000990_2023_business_report
스킵 (변경 없음, stat 일치): 000990_2023_governance_report
스킵 (변경 없음, stat 일치): 000990_2023_sustainability_report
스킵 (변경 없음, stat 일치): 000990_2024_business_report
스킵 (변경 없음, stat 일치): 000990_2024_governance_report
스킵 (변경 없음, stat 일치): 000990_2024_sustainability_report
스킵 (변경 없음, stat 일치): 000990_2025_business_report
스킵 (변경 없음, stat 일치): 000990_2025_governance_report
스킵 (변경 없음, stat 일치): 000990_2025_sustainability_report
스킵 (변경 없음, stat 일치): 007340_2022_business_report
스킵 (변경 없음, stat 일치): 007340_2022_governance_report
스킵 (변경 없음, stat 일치): 007340_2023_business_report
스킵 (변경 없음, stat 일치): 007340_2023_governance_report
스킵 (변경 없음, stat 일치): 007340_2023_sustainability_report
스킵 (변경 없음, stat 일치): 007340_2024_business_rep

In [29]:
# 수동 백업 및 안전 확인용 셀
backup_local_to_drive()
print("Drive 백업 완료!")

Drive 백업 완료!


In [13]:
# 로컬 Chroma가 실제로 정상인지 확인 (에러 없이 조회되면 정상)
test_vs = Chroma(persist_directory=CONFIG["persist_dir"], embedding_function=embeddings)
test_result = test_vs.similarity_search("테스트", k=1)
print(f"정상 작동: {len(test_result)}건 조회됨")

정상 작동: 1건 조회됨


In [34]:
import json
with open(CONFIG["checkpoint_log"], encoding="utf-8") as f:
    checkpoint = json.load(f)

failed_doc_ids = set(manifest.apply(
    lambda r: f"{r['company_code']}_{r['year']}_{r['doc_type']}", axis=1
)) - set(checkpoint.keys())
print(f"체크포인트에 없는(실패했거나 아직 처리 안 된) 문서: {len(failed_doc_ids)}건")
print(sorted(failed_doc_ids))

체크포인트에 없는(실패했거나 아직 처리 안 된) 문서: 0건
[]


In [ ]:
import shutil
from datetime import datetime

snapshot_dir = f"/content/drive/MyDrive/ESG프로젝트(개인)/db_snapshot_{datetime.now().strftime('%Y%m%d_%H%M')}"
shutil.copytree(CONFIG["persist_dir"], f"{snapshot_dir}/chroma_db")
shutil.copy(CONFIG["manifest_csv"], snapshot_dir)
shutil.copy(CONFIG["checkpoint_log"], snapshot_dir)
print(f"스냅샷 생성 완료: {snapshot_dir}")

스냅샷 생성 완료: /content/drive/MyDrive/ESG프로젝트(개인)/db_snapshot_20260902_1358


### [Cell 7b] (v4 신규) 코드 수정 후 재인덱싱 — 체크포인트 초기화
위 v4 패치(섹션/노이즈/표 fallback)를 반영한 뒤 다시 돌릴 때 필요.
PDF 파일 자체는 안 바뀌었으므로 Cell 7의 해시 체크가 그대로 통과해서 그냥 재실행하면 아무 일도 안 일어남.
**Cell 7을 먼저 실행하지 않아도 됨** — 이 셀이 checkpoint/vectorstore를 필요하면 직접 로드함.
실행 순서: Cell 1~6 → 이 셀(7b) → Cell 7 → Cell 8(export).

In [ ]:
# [v4 수정] checkpoint/vectorstore가 아직 메모리에 없을 수 있음
# (Cell 7을 이번 세션에서 한 번도 안 돌렸다면 당연히 NameError가 남).
# 그래서 없으면 파일/Drive에서 직접 로드하도록 독립적으로 만듦.
if "checkpoint" not in globals():
    if os.path.exists(CONFIG["checkpoint_log"]):
        with open(CONFIG["checkpoint_log"], encoding="utf-8") as f:
            checkpoint = json.load(f)
    else:
        checkpoint = {}
    print(f"checkpoint 새로 로드함 ({len(checkpoint)}건)")

if "vectorstore" not in globals():
    restore_local_from_drive_backup()  # [v5 신규] 로컬에 없으면 Drive 백업에서 복원
    _embeddings = GoogleGenerativeAIEmbeddings(model=CONFIG["embedding_model"])
    vectorstore = Chroma(
        persist_directory=CONFIG["persist_dir"],  # [v5] 로컬 경로
        embedding_function=_embeddings,
    )
    print("vectorstore 새로 로드함")

# 옵션 A) 특정 회사·문서만 재인덱싱 — 아래 리스트를 채워 넣을 것
# 예: _target_doc_ids = ["000660_2025_business_report", "005930_2025_governance_report"]
_target_doc_ids: list[str] = [
    # "기업코드_연도_문서유형",
]
_target_company_codes = {doc_id.split("_")[0] for doc_id in _target_doc_ids}

if not _target_doc_ids:
    print("⚠️ _target_doc_ids가 비어있음 — 재인덱싱할 doc_id를 채워 넣고 다시 실행할 것")
else:
    for _doc_id in _target_doc_ids:
        checkpoint.pop(_doc_id, None)
    for _company_code in _target_company_codes:
        vectorstore.delete(where={"company_code": _company_code})
    with open(CONFIG["checkpoint_log"], "w", encoding="utf-8") as f:
        json.dump(checkpoint, f, ensure_ascii=False, indent=2)
    print(f"체크포인트 초기화 완료: {_target_doc_ids} (Cell 7을 다시 실행할 것)")

# 옵션 B) 전체 강제 재인덱싱하려면 아래 두 줄만 사용 (위 옵션 A 대신):
# checkpoint = {}
# vectorstore.delete(where={})  # 주의: 전체 컬렉션 삭제


### [Cell 7c] (v5 신규) 텍스트 추출 실패 진단 CSV — OCR/LlamaParse 폴백 결정용 실측 데이터
Cell 7 루프가 끝난 뒤 실행. 회사·문서별로 추출 실패 의심 페이지 비율을 확인하고,
`issue_ratio`가 높은 문서(특히 `vector_curve_text`)가 몇 개사나 되는지 보고
OCR(Tesseract, 무료)와 LlamaParse(유료, 표 구조 보존) 중 무엇을 붙일지 결정할 것.

In [30]:
# [v5.1 수정] before(pdfplumber만) / after(캐스케이딩 후 잔여) 필드명으로 변경
diag_df = pd.DataFrame(extraction_diagnostics)
if diag_df.empty:
    print("진단 데이터 없음 (Cell 7을 먼저 실행할 것)")
else:
    diag_df.to_csv(CONFIG["extraction_diag_csv"], index=False, encoding="utf-8-sig")
    print(f"저장 완료: {CONFIG['extraction_diag_csv']} ({len(diag_df)}건)")

    print("\n=== 문서 유형별 평균 실패율 (before -> after) ===")
    print(diag_df.groupby("doc_type")[["issue_ratio_before", "issue_ratio_after"]]
          .agg(["mean", "max"]))

    print("\n=== 캐스케이딩 복구 현황 (문서 유형별 합계) ===")
    print(diag_df.groupby("doc_type")[["rescued_pymupdf", "rescued_llamaparse"]].sum())

    print("\n=== 캐스케이딩 후에도 잔여 실패가 있는 문서 (LlamaParse 재처리 후보) ===")
    bad = diag_df[diag_df["issue_pages_after"] > 0].sort_values("issue_ratio_after", ascending=False)
    print(bad[["company_code", "doc_type", "total_pages", "issue_pages_before",
               "rescued_pymupdf", "issue_pages_after", "issue_ratio_after", "still_failed_pages"]]
          .to_string(index=False) if not bad.empty else "없음 (LlamaParse 없이도 충분함)")
    print(f"\n-> 재처리 대상 문서 {len(bad)}건 / 총 잔여 실패 페이지 {bad['issue_pages_after'].sum() if not bad.empty else 0}건")
    print("   -> llamaparse_rescue.ipynb 로 이 CSV를 넣고 재처리할 것")

    print(f"\n지속가능경영보고서 중 캐스케이딩 후에도 심각(>50%) 실패 비율: "
          f"{(diag_df[diag_df['doc_type']=='sustainability_report']['issue_ratio_after'] > 0.5).mean():.1%}")


저장 완료: /content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/인덱스 결과/extraction_diagnostics.csv (100건)

=== 문서 유형별 평균 실패율 (before -> after) ===
                      issue_ratio_before        issue_ratio_after       
                                    mean    max              mean    max
doc_type                                                                
business_report                 0.005487  0.014          0.005487  0.014
governance_report               0.012387  0.125          0.011065  0.114
sustainability_report           0.204100  0.988          0.116833  0.988

=== 캐스케이딩 복구 현황 (문서 유형별 합계) ===
                       rescued_pymupdf  rescued_llamaparse
doc_type                                                  
business_report                      0                   0
governance_report                    4                   0
sustainability_report              279                   0

=== 캐스케이딩 후에도 잔여 실패가 있는 문서 (LlamaParse 재처리 후보) ===
company_code              doc_type  total_pages

In [31]:
# [진단 전용, 읽기만 함] 회사별 실패/누락 현황 확인
import pandas as pd, json

with open(CONFIG["checkpoint_log"], encoding="utf-8") as f:
    checkpoint = json.load(f)

manifest_check = pd.read_csv(CONFIG["manifest_csv"], dtype={"company_code": str})
manifest_check["doc_id"] = manifest_check.apply(
    lambda r: f"{r['company_code']}_{r['year']}_{r['doc_type']}", axis=1
)

expected_ids = set(manifest_check["doc_id"])
completed_ids = set(checkpoint.keys())
missing_ids = expected_ids - completed_ids

print(f"전체 대상: {len(expected_ids)}건 / 완료(checkpoint 기준): {len(completed_ids)}건 / "
      f"누락(실패 또는 아직 미처리): {len(missing_ids)}건\n")

# --- 1. 회사별 누락 현황 (실패했거나 아직 처리 안 된 것들) ---
missing_df = manifest_check[manifest_check["doc_id"].isin(missing_ids)]
if not missing_df.empty:
    company_summary = (
        missing_df.groupby("company_code")
        .agg(
            누락_건수=("doc_id", "count"),
            누락_문서유형=("doc_type", lambda s: sorted(s.tolist())),
            누락_연도=("year", lambda s: sorted(s.unique().tolist())),
        )
        .reset_index()
        .sort_values("누락_건수", ascending=False)
    )
    print("=== 회사별 누락 현황 ===")
    print(company_summary.to_string(index=False))
else:
    print("누락 없음 — manifest 전체가 checkpoint에 기록됨")

# --- 2. checkpoint엔 완료로 찍혔지만 실제 청크가 0개인 "가짜 완료" 탐지 ---
# (부분 적재된 채로 checkpoint만 기록된 경우 — 361610처럼 직접 강제 기록한 문서 등)
print("\n=== 체크포인트엔 완료로 찍혔지만 실제 Chroma엔 청크가 없는 문서 ===")
_zero_chunk_docs = []
for doc_id in completed_ids:
    result = vectorstore.get(where={"doc_id": doc_id}, limit=1)
    if not result["ids"]:
        _zero_chunk_docs.append(doc_id)

if _zero_chunk_docs:
    print(f"⚠️ {len(_zero_chunk_docs)}건 발견 — 실제로는 데이터가 없는데 완료 처리됨:")
    for d in _zero_chunk_docs:
        print(f"  {d}")
    print("\n이 문서들은 checkpoint에서 지우고 Cell 7을 다시 돌려야 재처리됨:")
    print("  for d in _zero_chunk_docs: checkpoint.pop(d, None)")
else:
    print("없음")

전체 대상: 450건 / 완료(checkpoint 기준): 450건 / 누락(실패 또는 아직 미처리): 0건

누락 없음 — manifest 전체가 checkpoint에 기록됨

=== 체크포인트엔 완료로 찍혔지만 실제 Chroma엔 청크가 없는 문서 ===
없음


In [32]:
# [진단 전용, 읽기만 함] LlamaParse 재처리 대상 — 페이지 수 기준으로 정리
import pandas as pd

diag_df = pd.read_csv(CONFIG["extraction_diag_csv"])

# still_failed_pages는 문자열(JSON 리스트)로 저장돼 있음
diag_df["still_failed_pages"] = diag_df["still_failed_pages"].apply(
    lambda v: json.loads(v) if pd.notna(v) and str(v).strip() else []
)
diag_df["잔여_실패_페이지수"] = diag_df["still_failed_pages"].apply(len)

# 잔여 실패가 있는 문서만
target_df = diag_df[diag_df["잔여_실패_페이지수"] > 0].copy()
target_df = target_df.sort_values("잔여_실패_페이지수", ascending=False)

print(f"=== 회사별 잔여 실패 페이지 (LlamaParse 대상) ===")
print(target_df[[
    "company_code", "doc_type", "total_pages",
    "issue_pages_before", "rescued_pymupdf", "잔여_실패_페이지수"
]].to_string(index=False))

print(f"\n총 재처리 대상 문서: {len(target_df)}건")
print(f"총 재처리 대상 페이지: {target_df['잔여_실패_페이지수'].sum()}페이지")

# 예상 크레딧/비용도 같이
_credits_per_page = 3 if CONFIG.get("llamaparse_mode", "cost_effective") == "cost_effective" else 1
_total_pages = target_df["잔여_실패_페이지수"].sum()
print(f"예상 크레딧: {_total_pages * _credits_per_page} (약 ${_total_pages * _credits_per_page * 0.00125:.2f})")

=== 회사별 잔여 실패 페이지 (LlamaParse 대상) ===
 company_code              doc_type  total_pages  issue_pages_before  rescued_pymupdf  잔여_실패_페이지수
         7810 sustainability_report           82                  81                0          81
         7810 sustainability_report           63                  62                0          62
       166090 sustainability_report           87                  25                7          18
        67310 sustainability_report           64                  27               13          14
        67310 sustainability_report           78                  27               14          13
         3670     governance_report           88                  11                1          10
       272210 sustainability_report          122                  21               12           9
         3670     governance_report           84                  10                1           9
       166090 sustainability_report           76                  26            

In [ ]:
# [검증 및 공유용] Chroma 인덱스를 CSV로 export.
# 100개사 전체를 export하면 파일이 매우 커질 수 있으니, 기본은 특정 회사
# 1개만 뽑도록 필터를 둠 — 전체를 뽑고 싶으면 EXPORT_COMPANY_CODE = None으로.
EXPORT_COMPANY_CODE = None  # 예: "005930" (특정 회사만) / None (전체)

where = {"company_code": EXPORT_COMPANY_CODE} if EXPORT_COMPANY_CODE else None
raw = vectorstore.get(where=where, include=["documents", "metadatas"])

export_rows = []
for doc_id_chroma, content, meta in zip(raw["ids"], raw["documents"], raw["metadatas"]):
    row = dict(meta)
    row["content"] = content
    export_rows.append(row)

export_df = pd.DataFrame(export_rows)

front_cols = [
    "company_code", "company_name", "year", "doc_type",
    "page", "section_title", "content_type", "chunk_id", "content",
]
other_cols = [c for c in export_df.columns if c not in front_cols]
export_df = export_df[front_cols + other_cols]

suffix = EXPORT_COMPANY_CODE if EXPORT_COMPANY_CODE else "ALL"
export_path = (
    "/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/인덱스 결과/"
    f"index_export_{suffix}.csv"
)
export_df.to_csv(export_path, index=False, encoding="utf-8-sig")
print(f"저장 완료: {export_path} ({len(export_df)}행)")
export_df.head()

저장 완료: /content/drive/MyDrive/ESG 팀프로젝트/3. RAG 인덱스/index_export_000660_SK하이닉스.csv (1996행)


,company_code,company_name,year,doc_type,page,section_title,content_type,chunk_id,content,doc_id,applicable_item_codes,source_file_hash,embedding_model,chunk_char_count,table_data,esg_area,indexed_at
0,000660,SK하이닉스,2025,business_report,1,,text,000660_2025_business_report_0001,목 차\n사 업 보 고 서...................................,000660_2025_business_report,[],f11f9c121fc80d110e53b92ba24fbad2,gemini-embedding-001,5388,,,2026-09-01T16:33:25.367774
1,000660,SK하이닉스,2025,business_report,1,,text,000660_2025_business_report_0002,12. 유형자산 (연결)....................................,000660_2025_business_report,[],f11f9c121fc80d110e53b92ba24fbad2,gemini-embedding-001,1301,,,2026-09-01T16:33:25.367774
2,000660,SK하이닉스,2025,business_report,2,,text,000660_2025_business_report_0003,"22. 자본금, 자본잉여금, 기타자본 및 기타포괄손익누계액 (연결)............",000660_2025_business_report,[],f11f9c121fc80d110e53b92ba24fbad2,gemini-embedding-001,4467,,,2026-09-01T16:33:25.367774
3,000660,SK하이닉스,2025,business_report,2,,text,000660_2025_business_report_0004,"10. 종속기업, 관계기업 및 공동기업투자..........................",000660_2025_business_report,[],f11f9c121fc80d110e53b92ba24fbad2,gemini-embedding-001,2332,,,2026-09-01T16:33:25.367774
4,000660,SK하이닉스,2025,business_report,3,,text,000660_2025_business_report_0005,28. 법인세비용(수익)....................................,000660_2025_business_report,[],f11f9c121fc80d110e53b92ba24fbad2,gemini-embedding-001,4812,,,2026-09-01T16:33:25.367774


### [Cell 8] 필터링 지원 하이브리드 리트리버

In [ ]:
# 핵심 변경: BM25를 전체 코퍼스로 한 번에 만들지 않고, 검색 시점의 필터
# (company_code 등)로 좁혀진 부분집합에서만 만든다. 회사별로 캐싱해서
# 같은 회사를 다시 물어볼 때는 재구축 없이 재사용.
class FilteredHybridRetriever:
    def __init__(self, vectorstore, tokenize_func, k=10, weights=(0.5, 0.5)):
        self.vectorstore = vectorstore
        self.tokenize_func = tokenize_func
        self.k = k
        self.weights = weights
        self._bm25_cache: dict[tuple, BM25Retriever] = {}

    def _get_bm25_retriever(self, filter_dict):
        sig = tuple(sorted((filter_dict or {}).items()))
        if sig in self._bm25_cache:
            return self._bm25_cache[sig]

        raw = self.vectorstore.get(
            where=filter_dict if filter_dict else None,
            include=["documents", "metadatas"],
        )
        docs = [
            Document(page_content=d, metadata=m)
            for d, m in zip(raw["documents"], raw["metadatas"])
        ]
        if not docs:
            self._bm25_cache[sig] = None
            return None

        retriever = BM25Retriever.from_documents(docs, preprocess_func=self.tokenize_func)
        retriever.k = self.k
        self._bm25_cache[sig] = retriever
        return retriever

    def invoke(self, query: str, filter: dict | None = None, top_n: int | None = None):
        top_n = top_n or self.k

        dense_docs = self.vectorstore.similarity_search(query, k=self.k, filter=filter)

        bm25_retriever = self._get_bm25_retriever(filter)
        bm25_docs = bm25_retriever.invoke(query) if bm25_retriever else []

        # Reciprocal Rank Fusion
        scores: dict[str, float] = {}
        lookup: dict[str, Document] = {}
        for rank, doc in enumerate(dense_docs):
            key = doc.metadata["chunk_id"]
            scores[key] = scores.get(key, 0.0) + self.weights[0] * (1.0 / (RRF_K + rank))
            lookup[key] = doc
        for rank, doc in enumerate(bm25_docs):
            key = doc.metadata["chunk_id"]
            scores[key] = scores.get(key, 0.0) + self.weights[1] * (1.0 / (RRF_K + rank))
            lookup[key] = doc

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return [lookup[k] for k, _ in ranked[:top_n]]

    def clear_cache(self):
        """회사 데이터를 재인덱싱한 뒤에는 캐시가 stale할 수 있으니 호출."""
        self._bm25_cache.clear()


hybrid_retriever = FilteredHybridRetriever(
    vectorstore=vectorstore,
    tokenize_func=kiwi_tokenize,
    k=CONFIG["retrieval_k"],
    weights=CONFIG["hybrid_weights"],
)

### [Cell 9] 동작 확인 — 반드시 실행해볼 것

In [ ]:
# manifest에 실제로 있는 회사 하나를 자동으로 골라서 테스트 (하드코딩 없음)
_test_company_code = manifest["company_code"].iloc[0]
print(f"테스트 대상 회사: {_test_company_code}")

# (1) dense 메타데이터 사전 필터 확인
test_filtered = vectorstore.similarity_search(
    "사외이사 비율", k=5, filter={"company_code": _test_company_code},
)
for d in test_filtered:
    assert d.metadata["company_code"] == _test_company_code, "필터링 실패 — 다른 회사 청크 혼입"
print(f"Dense 필터 테스트 통과: {len(test_filtered)}건")

# (2) 하이브리드 검색도 필터가 실제로 걸리는지 확인 (v2까지 안 되던 부분)
hybrid_results = hybrid_retriever.invoke(
    "이사회 산하 위원회 구성 현황", filter={"company_code": _test_company_code},
)
for d in hybrid_results:
    assert d.metadata["company_code"] == _test_company_code, "하이브리드 필터링 실패"
print(f"하이브리드 필터 테스트 통과: {len(hybrid_results)}건")
for r in hybrid_results[:3]:
    print(r.metadata["company_code"], r.metadata["doc_type"],
          r.metadata["page"], "|", r.page_content[:80])


# ==============================================================================

Dense 필터 테스트 통과: 5건
하이브리드 필터 테스트 통과: 10건
000660 governance_report 45 | 시
행
Y(O)
여
부
당사는 이사회의 효율적 운영과 전문성을 위하여 이사회 산하에 감사위원회, 사외이사후보추천위원회, 지속경영위원회, 인사·보
000660 business_report 316 | (주1) 2024년 4월 22일 감사위원회에서 한애라 사외이사를 감사위원회 위원장으로 신규 선임하였습니다.
(주2) 2021년 3월 30일 보상
000660 governance_report 46 | 명
문
규
Y(O)
정
여
부
당사는 이사회 내 위원회의 설치 목적, 조직, 운영, 권한 및 책임에 대해서는 각각의 위원회별 규정을 통해 서면으


### [Cell R] 세션 재시작 시 — 이 셀만 실행하면 됨 (Cell 1~4는 재실행 필요)

In [ ]:
# v2와 달리 전역 BM25를 pickle로 저장/복원하지 않으므로 로드가 훨씬 가벼움
# ==============================================================================
def load_existing_index():
    restore_local_from_drive_backup()  # [v5 신규] 로컬에 없으면 Drive 백업에서 복원
    _embeddings = GoogleGenerativeAIEmbeddings(model=CONFIG["embedding_model"])
    _vectorstore = Chroma(
        persist_directory=CONFIG["persist_dir"],  # [v5] 로컬 경로
        embedding_function=_embeddings,
    )
    _hybrid_retriever = FilteredHybridRetriever(
        vectorstore=_vectorstore,
        tokenize_func=kiwi_tokenize,
        k=CONFIG["retrieval_k"],
        weights=CONFIG["hybrid_weights"],
    )
    print("기존 인덱스 로드 완료 (BM25는 회사별로 필요할 때 자동 생성됨)")
    return _vectorstore, _hybrid_retriever


# 사용 예: vectorstore, hybrid_retriever = load_existing_index()